<a href="https://colab.research.google.com/github/edsonribferreira/Dashboard_EJA_EPT/blob/main/DadosEJA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import os

ARQUIVO_PRINCIPAL = 'dados_eja_completo.csv'
st.set_page_config(page_title="Dashboard EJA IFF Livre", layout="wide")

# --- FUNÇÕES DE DADOS ---
def salvar_dados(novo_df):
    if os.path.exists(ARQUIVO_PRINCIPAL):
        novo_df.to_csv(ARQUIVO_PRINCIPAL, mode='a', header=False, index=False)
    else:
        novo_df.to_csv(ARQUIVO_PRINCIPAL, index=False)

st.title("📊 Painel EJA - Análise Livre")

# --- ÁREA DE UPLOAD ---
with st.expander("⬆️ Alimentar Base de Dados"):
    novo_arquivo = st.file_uploader("Suba o CSV com os dados", type=['csv'])
    if novo_arquivo:
        salvar_dados(pd.read_csv(novo_arquivo))
        st.success("Dados integrados à base principal com sucesso!")

# --- ÁREA DE TESTES E CONFIGURAÇÕES ---
st.sidebar.divider()
st.sidebar.subheader("⚙️ Zona de Testes")

if st.sidebar.button("🗑️ Apagar Base de Dados (Reset)"):
    if os.path.exists(ARQUIVO_PRINCIPAL):
        os.remove(ARQUIVO_PRINCIPAL)
        st.sidebar.success("Arquivo de dados removido com sucesso!")
        # Força o Streamlit a recarregar a página para atualizar o dashboard
        st.rerun()
    else:
        st.sidebar.info("Não há dados para apagar.")

# --- APLICAÇÃO PRINCIPAL ---
if os.path.exists(ARQUIVO_PRINCIPAL):
    df = pd.read_csv(ARQUIVO_PRINCIPAL)

    # Remover colunas que não fazem sentido para agrupamento/gráficos (ex: nome)
    colunas_analise = [col for col in df.columns if col not in ['nome_estudante']]

    # --- 1. FILTROS GLOBAIS (BARRA LATERAL) ---
    st.sidebar.header("🔍 Filtros Globais")
    st.sidebar.write("Filtre a base antes de gerar o gráfico:")
    df_filtrado = df.copy()

    # Cria um filtro dinâmico para cada coluna do dataframe
    for col in colunas_analise:
        valores_unicos = df[col].dropna().unique()
        selecao = st.sidebar.multiselect(f"{col.capitalize()}", options=valores_unicos, default=[])
        if selecao:
            df_filtrado = df_filtrado[df_filtrado[col].isin(selecao)]

    # --- 2. CONSTRUTOR LIVRE DE GRÁFICOS ---
    st.header("🛠️ Construtor de Gráficos")

    c1, c2, c3 = st.columns(3)
    with c1:
        eixo_x = st.selectbox("1. Escolha a Categoria Principal (Eixo X):", colunas_analise)
    with c2:
        divisao_cor = st.selectbox("2. Cruzar com (Subdivisão/Cor):", ['Nenhum'] + colunas_analise)
    with c3:
        tipo_grafico = st.selectbox("3. Tipo de Gráfico:", ["Barras Agrupadas", "Barras Empilhadas", "Pizza / Donut"])

    st.divider()

    # --- 3. LÓGICA DE GERAÇÃO DO GRÁFICO ---
    if len(df_filtrado) == 0:
        st.warning("Nenhum dado encontrado para os filtros selecionados.")
    else:
        st.write(f"**Analisando {len(df_filtrado)} registros filtrados.**")

        # Cenário 1: Cruzando duas colunas (Ex: Curso vs Renda)
        if divisao_cor != 'Nenhum':
            # Agrupa e conta a quantidade
            df_agrupado = df_filtrado.groupby([eixo_x, divisao_cor]).size().reset_index(name='Quantidade')

            if tipo_grafico == "Barras Agrupadas":
                fig = px.bar(df_agrupado, x=eixo_x, y='Quantidade', color=divisao_cor, barmode='group')
            elif tipo_grafico == "Barras Empilhadas":
                fig = px.bar(df_agrupado, x=eixo_x, y='Quantidade', color=divisao_cor)
            else: # Pizza não suporta bem cruzamento duplo natural sem facets, mas podemos fazer um Sunburst ou Pizza simples do agrupamento
                fig = px.sunburst(df_agrupado, path=[eixo_x, divisao_cor], values='Quantidade')

        # Cenário 2: Analisando apenas uma coluna (Ex: Apenas Etnia)
        else:
            df_agrupado = df_filtrado[eixo_x].value_counts().reset_index()
            df_agrupado.columns = [eixo_x, 'Quantidade']

            if tipo_grafico in ["Barras Agrupadas", "Barras Empilhadas"]:
                fig = px.bar(df_agrupado, x=eixo_x, y='Quantidade', color=eixo_x)
            else:
                fig = px.pie(df_agrupado, names=eixo_x, values='Quantidade', hole=0.4)

        # Exibe o gráfico gerado
        st.plotly_chart(fig, use_container_width=True)

    # --- 4. TABELA DE DADOS RAW ---
    with st.expander("Ver Tabela de Dados Filtrados"):
        st.dataframe(df_filtrado)

else:
    st.info("Nenhum dado no sistema. Faça o upload do primeiro arquivo CSV.")